In [0]:
from pyspark.sql.functions import col, lower

# 1. Lendo as tabelas Delta da camada Bronze (usamos spark.table agora!)
df_clientes_bronze = spark.table("workspace.default.bronze_clientes")
df_produtos_bronze = spark.table("workspace.default.bronze_produtos")
df_vendas_bronze = spark.table("workspace.default.bronze_vendas_logistica")

# 2. Aplicando regras de qualidade de dados (Data Quality) - Camada Silver

# Tratando Clientes: Apenas removendo possíveis duplicatas
df_clientes_silver = df_clientes_bronze.dropDuplicates()

# Tratando Produtos: Removendo duplicatas
df_produtos_silver = df_produtos_bronze.dropDuplicates()

# Tratando Vendas: 
# - Removendo duplicatas
# - Preenchendo valores vazios (nulos) de frete com 0.0
# - Padronizando a coluna Status_Entrega para tudo minúsculo
df_vendas_silver = df_vendas_bronze.dropDuplicates() \
    .fillna({'Custo_Frete': 0.0}) \
    .withColumn("Status_Entrega", lower(col("Status_Entrega")))

# 3. Salvando as tabelas limpas na camada Silver do Unity Catalog
df_clientes_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_clientes")
df_produtos_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_produtos")
df_vendas_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_vendas_logistica")

print("Processamento Silver concluído! Dados limpos, padronizados e salvos com sucesso.")

Processamento Silver concluído! Dados limpos, padronizados e salvos com sucesso.
